# Day 3: Deterministic single object optimization
---
### The goal:
    We have a situation. There is a choice, or a set of choices we need to make, in order to optimise a certain parameter. Here, we focus on optimising a single decision variable, with a set of constraine defined already. That gives the name "Deterministic".

In [3]:
import pulp

# Two shovels need material moved. Trucks can be assigned to each shovel.
# Decision variables: number of trucks assigned to shovel A and shovel B.

prob = pulp.LpProblem("Truck_Assignment", pulp.LpMaximize)

x_A = pulp.LpVariable("trucks_to_shovel_A", lowBound=0, cat="Integer")
x_B = pulp.LpVariable("trucks_to_shovel_B", lowBound=0, cat="Integer")

# Productivity per truck per shift
tons_A = 900
tons_B = 750

# Objective: maximize total tons moved
prob += tons_A * x_A + tons_B * x_B

# Constraints
prob += x_A + x_B <= 6, "available_trucks"
prob += x_A <= 4, "max_trucks_shovel_A"
prob += x_B <= 4, "max_trucks_shovel_B"
prob += tons_A * x_A >= 1800, "minimum_shovel_A_tons"
prob += tons_B * x_B >= 1500, "minimum_shovel_B_tons"

# Solving the optimisation problem
prob.solve(pulp.PULP_CBC_CMD(msg=False))

print("Status:", pulp.LpStatus[prob.status])
print("Trucks to Shovel A:", x_A.value())
print("Trucks to Shovel B:", x_B.value())
print("Total tons:", pulp.value(prob.objective))

Status: Optimal
Trucks to Shovel A: 4.0
Trucks to Shovel B: 2.0
Total tons: 5100.0


## Result
With all the constraints, the most optimal solution is to have 4 trucks at shovel A and 2 trucks at shovel B. This way the total production will be 5100 tons.
___
# Exercise
We need to change the decision variable from maximizing production to minimizing cost.
The problem model will change accordingly

In [38]:
import pulp

# Two shovels need material moved. Trucks can be assigned to each shovel.
# Decision variables: number of trucks assigned to shovel A and shovel B.
# Goal is to minimise the total operating costs

prob = pulp.LpProblem("Truck_Assignment", pulp.LpMinimize)

x_A = pulp.LpVariable("trucks_to_shovel_A", lowBound=0, cat="Integer")
x_B = pulp.LpVariable("trucks_to_shovel_B", lowBound=0, cat="Integer")

# Cost per truck per shift
cost_A = 1000
cost_B = 850

# Productivity per truck per shift
tons_A = 900
tons_B = 750

# Objective: minimise total operating costs
prob += cost_A * x_A + cost_B * x_B

# Constraints

# If we want to have the costs minimised,
# we need to put a lower limit on factors directly proportional to costs,
# and a higher bound on things that are inversely proportional to costs. 

# what would you do to reduce costs? have less production, right? lets put a constraint on that
# total minimum production requirement:
prob += tons_A * x_A + tons_B * x_B >= 5800


# Since we need to minimise costs, having an upper limit on fleet size is probably redundant,
# but it makes the model more representative of real world scenario where trucks are limited
# Lets experiment if it changes anything or not
prob += x_A + x_B <= 8, "available_trucks"

# Is having a constraint of number of trucks per shovel essential? 
# No, atleast in the ideal world. But could be, due to other real factors, 
# like queue length? too many trucks at one shovel may explode queue time. 
# Considering queue time, having a upper limit per shovel is sensible.
# There could be other factors as well, not accounted in this problem.
#prob += x_A <= 4, "max_trucks_shovel_A"
#prob += x_B <= 4, "max_trucks_shovel_B"
#prob += tons_A * x_A >= 100, "minimum_shovel_A_tons"
#prob += tons_B * x_B >= 1500, "minimum_shovel_B_tons"

# I did some experiment with some values. Having too many constraints makes the system infeasable in that range
# For example, you have max 6 trucks and production is 900 and 750 for two shovels. you give production target 5300,
# and put a limit of 4 on max truck per shovel. in this case, the max production would limit to 5100 only,
# and youre left with no space to optmise.
# I have kept only the min production, and max trucks constraints, to allow the system to optimise, 
# Ofcourse, only to have an understanding of optimisation.

# Solving the optimisation problem
prob.solve(pulp.PULP_CBC_CMD(msg=False))

print("Status:", pulp.LpStatus[prob.status])
print("Trucks to Shovel A:", x_A.value())
print("Trucks to Shovel B:", x_B.value())
print("Minimum Cost:", pulp.value(prob.objective))
print("Production:", tons_A * x_A.value() + tons_B * x_B.value())

Status: Optimal
Trucks to Shovel A: 4.0
Trucks to Shovel B: 3.0
Minimum Cost: 6550.0
Production: 5850.0


## Results. 
By routing 4 trucks to shovel A and 3 to B, we achive the optimal condition, having a minimum cost of 6550 and production of 5850.
we had a MIN Production requriemtn of 5800 tons. 

Comments are added in the code, to explain observations noted while experimenting with constraints and constraint values. 